# 05 Final Load Prep

This notebook prepares the final dataset for downstream use:
1. Feature engineering — Month, Day of Week, Price Bucket
2. Column renaming to snake_case
3. Drop is_price_outlier column
4. Final validation checks
5. Save final dataset to `data/processed/swiggy_final.csv`

In [11]:
from pathlib import Path

import pandas as pd

current_dir = Path.cwd().resolve()
PROJECT_ROOT = current_dir.parent if current_dir.name.strip() == 'notebooks' else current_dir

In [12]:
PROCESSED_PATH = PROJECT_ROOT / 'data/processed/swiggy_cleaned.csv'
df = pd.read_csv(PROCESSED_PATH, parse_dates=['order_date'])
print('Loaded:', df.shape)
df.head()

Loaded: (91788, 14)


,state,city,order_date,restaurant_name,location,category,dish_name,price_inr,rating,rating_count,is_price_outlier,month,day_of_week,price_bucket
0,Karnataka,Bengaluru,2025-04-03,Srinidhi Sagar Deluxe,Kengeri,Recommended,Badam Milk,52.0,4.5,25,False,2025-04,Thursday,Budget
1,Karnataka,Bengaluru,2025-01-15,Srinidhi Sagar Deluxe,Kengeri,Recommended,Chow Chow Bath,117.0,4.7,48,False,2025-01,Wednesday,Budget
2,Karnataka,Bengaluru,2025-01-21,Srinidhi Sagar Deluxe,Kengeri,Recommended,Garlic Naan,98.0,4.0,34,False,2025-01,Tuesday,Budget
3,Karnataka,Bengaluru,2025-05-02,Srinidhi Sagar Deluxe,Kengeri,North Indian Gravy,Panneer Butter Masala,241.0,4.4,29,False,2025-05,Friday,Mid-range
4,Karnataka,Bengaluru,2025-07-30,Srinidhi Sagar Deluxe,Kengeri,North Indian Gravy,Dal Tadka,195.0,4.9,51,False,2025-07,Wednesday,Mid-range


## 1. Feature Engineering

In [13]:
df['month'] = df['order_date'].dt.to_period('M').astype(str)
df['day_of_week'] = df['order_date'].dt.day_name()
print('Month and Day of Week columns added.')

Month and Day of Week columns added.


In [14]:
bins   = [0, 150, 400, float('inf')]
labels = ['Budget', 'Mid-range', 'Premium']
df['price_bucket'] = pd.cut(df['price_inr'], bins=bins, labels=labels, right=False)
print('Price Bucket distribution:')
print(df['price_bucket'].value_counts())

Price Bucket distribution:
price_bucket
Mid-range    54246
Budget       26915
Premium      10627
Name: count, dtype: int64


## 2. Drop is_price_outlier Column

In [15]:
df = df.drop(columns=['is_price_outlier'])
print('is_price_outlier column dropped.')

is_price_outlier column dropped.


## 3. Rename Columns to snake_case

In [16]:
rename_map = {
    'state'           : 'state',
    'city'            : 'city',
    'order_date'      : 'order_date',
    'restaurant_name' : 'restaurant_name',
    'location'        : 'location',
    'category'        : 'category',
    'dish_name'       : 'dish_name',
    'price_inr'     : 'price_inr',
    'rating'          : 'rating',
    'rating_count'    : 'rating_count',
    'month'           : 'month',
    'day_of_week'     : 'day_of_week',
    'price_bucket'    : 'price_bucket'
}
df = df.rename(columns=rename_map)
print('Columns renamed to snake_case.')
print('Final columns:', df.columns.tolist())

Columns renamed to snake_case.
Final columns: ['state', 'city', 'order_date', 'restaurant_name', 'location', 'category', 'dish_name', 'price_inr', 'rating', 'rating_count', 'month', 'day_of_week', 'price_bucket']


## 4. Final Validation Checks

In [17]:
print('===== Final Validation =====')

null_count = df.isnull().sum().sum()
print(f'Null values      : {null_count} {"✓" if null_count == 0 else "✗ — check nulls"}')

dup_count = df.duplicated().sum()
print(f'Duplicate rows   : {dup_count} {"✓" if dup_count == 0 else "✗ — check duplicates"}')

print(f'Final row count  : {len(df):,}')
print(f'Final col count  : {len(df.columns)}')
print(f'Columns          : {df.columns.tolist()}')

===== Final Validation =====
Null values      : 0 ✓
Duplicate rows   : 0 ✓
Final row count  : 91,788
Final col count  : 13
Columns          : ['state', 'city', 'order_date', 'restaurant_name', 'location', 'category', 'dish_name', 'price_inr', 'rating', 'rating_count', 'month', 'day_of_week', 'price_bucket']


## 5. Save Final Dataset

In [18]:
FINAL_PATH = PROJECT_ROOT / 'data/processed/swiggy_final_cleaned.csv'
df.to_csv(FINAL_PATH, index=False)
print(f'Final dataset saved to: {FINAL_PATH}')
df.head()

Final dataset saved to: /Users/kshitijsaxena/Desktop/Hopper_C-2_SwiggyDataAnalysis/data/processed/swiggy_final_cleaned.csv


,state,city,order_date,restaurant_name,location,category,dish_name,price_inr,rating,rating_count,month,day_of_week,price_bucket
0,Karnataka,Bengaluru,2025-04-03,Srinidhi Sagar Deluxe,Kengeri,Recommended,Badam Milk,52.0,4.5,25,2025-04,Thursday,Budget
1,Karnataka,Bengaluru,2025-01-15,Srinidhi Sagar Deluxe,Kengeri,Recommended,Chow Chow Bath,117.0,4.7,48,2025-01,Wednesday,Budget
2,Karnataka,Bengaluru,2025-01-21,Srinidhi Sagar Deluxe,Kengeri,Recommended,Garlic Naan,98.0,4.0,34,2025-01,Tuesday,Budget
3,Karnataka,Bengaluru,2025-05-02,Srinidhi Sagar Deluxe,Kengeri,North Indian Gravy,Panneer Butter Masala,241.0,4.4,29,2025-05,Friday,Mid-range
4,Karnataka,Bengaluru,2025-07-30,Srinidhi Sagar Deluxe,Kengeri,North Indian Gravy,Dal Tadka,195.0,4.9,51,2025-07,Wednesday,Mid-range
